In [1]:
import numpy as np

import transpose_invariance as tpi

import skimage as ski
import skimage.segmentation as sks

Adapted from:
https://scikit-image.org/docs/stable/auto_examples/segmentation/plot_random_walker_segmentation.html

In [2]:
rng = np.random.default_rng()

# Generate noisy synthetic data
bb = ski.data.binary_blobs(length=32, n_dim=3, rng=rng)
img = ski.img_as_float(bb)
sigma = 0.35
img += rng.normal(loc=0, scale=sigma, size=img.shape)
img = ski.exposure.rescale_intensity(img,
                                     in_range=(-sigma, 1 + sigma),
                                     out_range=(-1, 1))

# The range of the binary image spans over (-1, 1).
# We choose the hottest and the coldest pixels as markers.
markers = np.zeros(img.shape, dtype=np.uint)
markers[img < -0.95] = 1
markers[img > 0.95] = 2

/Volumes/zorg/mb312/dev_trees/coordinate-review/main/src/skimage/data/_binary_blobs.py:3: ExperimentalAPIWarning: Importing from the `skimage2` namespace is experimental. Its API is under development and considered unstable!
  import skimage2 as ski2


In [3]:
img.shape

(32, 32, 32)

In [4]:
def func(img, markers):
    return sks.random_walker(img, markers)

ws_orig = func(img, markers)

/Volumes/zorg/mb312/dev_trees/coordinate-review/main/src/skimage/_shared/utils.py:604: UserWarning: The probability range is outside [0, 1] given the tolerance `prob_tol`. Consider decreasing `beta` and/or decreasing `tol`.
  return func(*args, **kwargs)


In [5]:
def rolled_proc_markers(img, markers, axes, func):
    r_img = np.transpose(img, axes)
    r_markers = np.transpose(markers, axes)
    f_r_img = func(r_img, r_markers)
    return np.transpose(f_r_img, np.argsort(axes))

In [6]:
for order in tpi.orderings:
    print('Testing', order)
    ws_rolled = rolled_proc_markers(img, markers, order, func)
    tpi.assert_labels_equivalent(ws_rolled, ws_orig)

Testing (0, 2, 1)
Testing (1, 2, 0)
Testing (2, 1, 0)
Testing (2, 0, 1)
Testing (1, 0, 2)
